### Story 618 — Perform basic geometry checks when adding/updating items in the catalog
  * https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-618

This story adds **server-side validation and enforcement** for `geometry` and `bbox` when **creating or updating STAC Items** via:

- `POST  /catalog/collections/{ownerId:collectionId}/items`
- `PUT   /catalog/collections/{ownerId:collectionId}/items/{featureId}`
- `PATCH /catalog/collections/{ownerId:collectionId}/items/{featureId}`

It implements two ESA/STAC requirements:

- **STAC-CORE-ITEM-REQ-0220 — Polygon Geometry**: Polygon/MultiPolygon rings must follow the **right-hand rule** (exterior ring **CCW**, interior rings **CW**), per RFC 7946.
- **STAC-CORE-ITEM-REQ-0230 — Minimum-bounding rectangle**: `bbox` must be **2D** and formatted as **4 values**: `[minLon, minLat, maxLon, maxLat]` (**SW → NE** order).

#### Accepted cases

- **`geometry = null` AND `bbox = null`/missing**  
  Accepted (e.g., CADIP sessions). **No geometry checks performed**.

#### Auto-enforcement (server modifies the item)

- **`geometry` present AND `bbox` missing**  
  The server **computes** `bbox` from the geometry bounds and adds it.

- **`geometry` present AND `bbox` present**  
  The server enforces that `bbox` is **consistent with geometry bounds** (strict mode). If consistent, it normalizes numbers to floats.

#### Rejected cases (HTTP 400 + clear message; item is NOT added/updated)

- `geometry = null` but `bbox` is provided →  
  `"Invalid STAC item: bbox provided but geometry is null."`

- `geometry` is not a GeoJSON object (`dict`) →  
  `"Invalid GeoJSON geometry: expected an object."`

- `geometry` cannot be parsed by Shapely (`shape(...)` fails) →  
  `"Invalid GeoJSON geometry: {exception_message}"`

- `geometry` is empty →  
  `"Invalid GeoJSON geometry: empty geometry is not allowed."`

- `geometry` is topologically invalid (`is_valid == False`, e.g., self-intersection) →  
  `"Invalid GeoJSON geometry: {shapely_reason}."`

- `MultiPolygon` has invalid `coordinates` type →  
  `"Invalid GeoJSON MultiPolygon: coordinates must be an array."`

- Polygon/MultiPolygon ring structure / validity errors:

  - No rings / rings not an array →  
    `"Invalid {Polygon|MultiPolygon[i]}: expected at least one linear ring."`

  - Ring is not an array →  
    `"Invalid {ring_label}: ring must be an array of positions."`

  - Ring has < 4 positions →  
    `"Invalid {ring_label}: ring must contain at least 4 positions."`

  - Ring is not closed (first != last) →  
    `"Invalid {ring_label}: ring must be closed (first and last positions must match)."`

  - Degenerate ring (area = 0) →  
    `"Invalid {ring_label}: degenerate ring area is zero."`

  - Wrong orientation (right-hand rule) →  
    `"Invalid {ring_label}: expected {counterclockwise|clockwise} orientation (right-hand rule)."`

  - A position is not at least `[lon, lat]` →  
    `"Invalid {position_label}: expected at least [lon, lat]."`

- `bbox` validation errors (REQ-0230):

  - `bbox` not an array →  
    `"Invalid bbox: expected an array."`

  - `bbox` length != 4 →  
    `"Invalid bbox: expected an array of length 4 [minLon, minLat, maxLon, maxLat]."`

  - `bbox` SW/NE order invalid (`minLon > maxLon` or `minLat > maxLat`) →  
    `"Invalid bbox: expected southwesterly point followed by northeasterly point."`

  - `bbox` contains non-numeric / boolean values →  
    `"Invalid numeric value in bbox."`

  - `bbox` does not match geometry bounds (strict consistency) →  
    `"Inconsistent bbox for geometry. Expected {expected_bbox}, got {parsed_bbox}."`

In [1]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *
import pprint

init_demo()
init_dask_cluster_staging(scale=2)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  

pp = pprint.PrettyPrinter(indent=2, width=80, sort_dicts=False, compact=True)

# You can check here the number of workers, threads and memory per worker.
# In local mode, you can configure them by running e.g.
# DASK_MEMORY_EOPF=4G DASK_CORES_EOPF=4 docker compose up # ...
display(dask_cluster_staging)

Auxip service: http://rs-server-adgs:8000/auxip
PRIP service: http://rs-server-prip:8000/prip
CADIP service: http://rs-server-cadip:8000/cadip
EDRS service: http://rs-server-edrs:8000/edrs
Catalog service: http://rs-server-catalog:8000
Staging service: http://rs-server-staging:8000
DPR service: http://rs-dpr-service:8000
OSAM service: http://rs-server-osam:8000
Connecting to dask gateway for 'dask-staging': http://dask-staging:8000 ...
Create new dask cluster
Dask dashboard for 'dask-staging': http://localhost:8701/clusters/707b45834dcc4e8f881d3d9470d2dfc9/status


/opt/conda/lib/python3.13/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| lz4     | 4.4.5  | None      | None    |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


Dask workers for 'dask-staging' are up: 0/2
Dask workers for 'dask-staging' are up: 2/2


In [2]:
# Create a test collection
CATALOG_COLLECTION_ID = "SPRINT_34_RSPY_618_TEST_COLLECTION"
collection = create_test_collection(CATALOG_COLLECTION_ID)
items = catalog_client.get_items(CATALOG_COLLECTION_ID)
list(items)

14:59:30.353 [INFO] (rs_client.rs_client) Retrieving all items from collection 'abutu:SPRINT_34_RSPY_618_TEST_COLLECTION'.


[]

In [3]:
cadip_no_geometry_no_bbox = cadip_client.search(
    method="GET", 
    stac_filter="externalIds=cadip:6f3c8d91-2b0e-492d-aef6-87b24f2bcb1e")
cadip_no_geometry_no_bbox

In [4]:
# 1. 
# Insert a CADIP session STAC item without geometry nor bbox. Check that the item is successfully inserted.
fc = cadip_no_geometry_no_bbox.to_dict()
item_dict = fc["features"][0]
item_dict["collection"] = CATALOG_COLLECTION_ID
item_dict.setdefault("stac_extensions", [])
item_dict["assets"] = {}  # Avoid triggering the bucket-transfer logic (the temp S3 copy/move of assets)

owner = catalog_client.owner_id
url = f"{catalog_client.href_service}/catalog/collections/{owner}:{CATALOG_COLLECTION_ID}/items"

if cluster_mode:
    resp = http_session.post(url, json=item_dict, headers={"x-api-key": os.environ["RSPY_APIKEY"]}, timeout=120)
else:
    resp = http_session.post(url, json=item_dict, timeout=120)

print(resp.status_code)
print(resp.text)


201
{"bbox":null,"type":"Feature","geometry":null,"properties":{"datetime":"2021-04-10T03:39:28.012000Z","updated":"2026-03-03T14:59:32.159612Z","start_datetime":"2021-04-10T03:19:28.012000Z","end_datetime":"2021-04-10T03:40:06.012000Z","platform":"sentinel-1a","constellation":"sentinel-1","published":"2026-03-03T14:59:32.159573Z","cadip:num_channels":2,"cadip:station_unit_id":"01","sat:absolute_orbit":53186,"cadip:acquisition_id":"53186_A1","cadip:antenna_id":"MSP21","cadip:front_end_id":"01","cadip:retransfer":false,"cadip:antenna_status_ok":true,"cadip:front_end_status_ok":true,"cadip:planned_data_start":"2021-04-10T03:19:28.012Z","cadip:planned_data_stop":"2021-04-10T03:29:28.012Z","cadip:downlink_status_ok":true,"cadip:delivery_push_ok":true,"externalIds":[{"scheme":"cadip","value":"6f3c8d91-2b0e-492d-aef6-87b24f2bcb1e"}],"eopf:origin_datetime":"2021-04-10T03:40:06.012Z","owner":"abutu","expires":"2026-04-02T14:59:32.166974Z"},"id":"S1A_20210410031928012345","stac_version":"1.1.0"

In [5]:
import copy, pystac

In [6]:
result = list(catalog_client.get_collection(CATALOG_COLLECTION_ID).get_items())
result

[<Item id=S1A_20210410031928012345>]

In [7]:
item_collection_prip = prip_client.search(
    method='GET',
    stac_filter="externalIds=prip:4db05e5e-16d7-4c15-8ca1-9a7d31d06eba, b99c8f80-ee84-4854-bd36-15b18ac0ecca")
item_collection_prip

In [8]:
# 2.
# Insert an item with an invalid geojson geometry. 
# Check that HTTP 400 bad request error is returned with a clear error message and that the item is NOT added to the catalog.
item = copy.deepcopy(item_collection_prip.to_dict()['features'][0])
item["collection"] = CATALOG_COLLECTION_ID
# invalid geojson geometry (not an object)
item["geometry"] = "not-an-object"
item.pop("bbox", None)  # 

owner = catalog_client.owner_id
url = f"{catalog_client.href_service}/catalog/collections/{owner}:{CATALOG_COLLECTION_ID}/items"
r = http_session.post(url, json=item, timeout=120)
print(r.status_code)
print(r.text)  # error message

400
{"code":"BadRequest","description":"Invalid GeoJSON geometry: expected an object."}


In [9]:
# 2.
# Insert an item with an invalid geojson geometry. 
# Check that HTTP 400 bad request error is returned with a clear error message and that the item is NOT added to the catalog.
item = copy.deepcopy(item_collection_prip.to_dict()['features'][0])
item["collection"] = CATALOG_COLLECTION_ID
# invalid geojson geometry (ring not closed)
item["geometry"] = {"type": "Polygon", "coordinates": [[[0,0],[1,0],[1,1],[0,1]]]}
item.pop("bbox", None)  #

owner = catalog_client.owner_id
url = f"{catalog_client.href_service}/catalog/collections/{owner}:{CATALOG_COLLECTION_ID}/items"
r = http_session.post(url, json=item, timeout=120)
print(r.status_code)
print(r.text)  # error message

400
{"code":"BadRequest","description":"Invalid Polygon exterior ring: ring must be closed (first and last positions must match)."}


In [10]:
# 3.
# Insert an item with a valid geojson geometry but no bbox.
# Check that RS-Server computes and adds the bbox as per STAC-CORE-ITEM-REQ-0230 requirement.
item = copy.deepcopy(item_collection_prip.to_dict()['features'][0])
item["collection"] = CATALOG_COLLECTION_ID
# invalid geojson geometry (nu e obiect)
item.pop("bbox", None)  # no bbox
item["assets"] = {}  # Avoid triggering the bucket-transfer logic (the temp S3 copy/move of assets)

owner = catalog_client.owner_id
url = f"{catalog_client.href_service}/catalog/collections/{owner}:{CATALOG_COLLECTION_ID}/items"
r = http_session.post(url, json=item, timeout=120)
print(r.status_code)
print(r.text)  # 

201
{"bbox":[101.385001349224,79.6835150322236,103.900525677419,80.0894153951246],"type":"Feature","geometry":{"type":"Polygon","coordinates":[[[103.080911931087,80.0894153951246],[101.385001349224,79.8622675367147],[102.217935425109,79.6835150322236],[103.900525677419,79.9060170135641],[103.080911931087,80.0894153951246]]]},"properties":{"datetime":"2025-08-01T07:06:20.464000Z","created":"2025-08-01T07:26:48Z","updated":"2026-03-03T14:59:34.614307Z","start_datetime":"2025-08-01T07:06:20.464000Z","end_datetime":"2025-08-01T07:06:20.464000Z","platform":"sentinel-2a","instruments":["SAR"],"constellation":"sentinel-2","published":"2026-03-03T14:59:34.614239Z","expires":"2025-08-08T07:45:53.277Z","eopf:origin_datetime":"2025-08-01T07:26:48Z","product:type":"OPER_MSI","product:timeliness":null,"product:timeliness_category":null,"processing:lineage":"systematic_production","processing:datetime":"2025-08-01T07:26:48Z","processing:facility":"S2 Production Service-SERCO","processing:level":"L0"

In [11]:
# 4.
# Try to replace the contents of a valid item with an invalid geojson geometry using PUT. 
# Check that HTTP 400 bad request error is returned with a clear error message and that the item is NOT modified in the catalog.

import copy, pystac

COLL = CATALOG_COLLECTION_ID
owner = catalog_client.owner_id

# 1) 
item_id = "S2B_OPER_MSI_L0__GR_2BPS_20250801T074015_S20250801T070620_D04_N05.11"  # id
original = catalog_client.get_item(COLL, item_id).to_dict()

# 2) (invalid GeoJSON)
bad = copy.deepcopy(original)
# bad["geometry"] = {"type": "Polygon", "coordinates": []}  # invalid
ring = original["geometry"]["coordinates"][0]
rev = ring[:-1][::-1]
bad["geometry"] = {"type": "Polygon", "coordinates": [rev + [rev[0]]]}


# 3) PUT (replace)
url = f"{catalog_client.href_service}/catalog/collections/{owner}:{COLL}/items/{item_id}"
resp = http_session.put(url, json=bad, timeout=120)

print(resp.status_code)
print(resp.text)  # 400

# 4) not modified in catalog
after = catalog_client.get_item(COLL, item_id).to_dict()
assert after["geometry"] == original["geometry"]
assert after.get("bbox") == original.get("bbox")


400
{"code":"BadRequest","description":"Invalid Polygon exterior ring: expected counterclockwise (CCW) orientation (right-hand rule)."}


In [12]:
# 5. Same as above with PATCH.
# Try to replace the contents of a valid item with an invalid geojson geometry using PATCH. 
# Check that HTTP 400 bad request error is returned with a clear error message and that the item is NOT modified in the catalog.

owner = catalog_client.owner_id
url = f"{catalog_client.href_service}/catalog/collections/{owner}:{COLL}/items/{item_id}"

payload = {"geometry": {"type": "Polygon", "coordinates": []}, "properties": {}}
r = http_session.patch(url, json=payload, timeout=120)

print(r.status_code)
print(r.text)

400
{"code":"BadRequest","description":"Invalid GeoJSON geometry: empty geometry is not allowed."}


In [13]:
# 6. 
# Try to remove bbox of a valid item using PUT. 
# Check that RS-Server computes again the bbox and adds it back, so that the item in catalog still has a bbox.
import copy

owner = catalog_client.owner_id
url = f"{catalog_client.href_service}/catalog/collections/{owner}:{COLL}/items/{item_id}"

original = catalog_client.get_item(COLL, item_id).to_dict()

# PUT - bbox removed
modified = copy.deepcopy(original)
modified.pop("bbox", None)

r = http_session.put(url, json=modified, timeout=120)
print(r.status_code, r.text)

# bbox was re-computed
after = catalog_client.get_item(COLL, item_id).to_dict()
assert after["bbox"] is not None
print(after["bbox"])


200 {"bbox":[101.385001349224,79.6835150322236,103.900525677419,80.0894153951246],"type":"Feature","geometry":{"type":"Polygon","coordinates":[[[103.080911931087,80.0894153951246],[101.385001349224,79.8622675367147],[102.217935425109,79.6835150322236],[103.900525677419,79.9060170135641],[103.080911931087,80.0894153951246]]]},"properties":{"datetime":"2025-08-01T07:06:20.464000Z","created":"2025-08-01T07:26:48Z","updated":"2026-03-03T14:59:35.085872Z","start_datetime":"2025-08-01T07:06:20.464000Z","end_datetime":"2025-08-01T07:06:20.464000Z","platform":"sentinel-2a","instruments":["SAR"],"constellation":"sentinel-2","owner":"abutu","expires":"2025-08-08T07:45:53.277Z","published":"2026-03-03T14:59:34.614239Z","externalIds":[{"value":"b99c8f80-ee84-4854-bd36-15b18ac0ecca","scheme":"prip"}],"product:type":"OPER_MSI","sat:orbit_cycle":265,"sat:orbit_state":"ASCENDING","eopf:datatake_id":342155,"processing:level":"L0","sar:polarizations":"HH","processing:lineage":"systematic_production","sa

In [14]:
# Try to remove bbox of a valid item using PATCH.
# Check that RS-Server computes again the bbox and adds it back, so that the item in catalog still has a bbox.
owner = catalog_client.owner_id
url = f"{catalog_client.href_service}/catalog/collections/{owner}:{COLL}/items/{item_id}"

# PATCH: remove bbox (middleware recomputes from geometry)
payload = {"bbox": None, "properties": {}}
r = http_session.patch(url, json=payload, timeout=120)
print(r.status_code, r.text)

after = catalog_client.get_item(COLL, item_id).to_dict()
assert after["bbox"] is not None
print(after["bbox"])


200 {"id":"S2B_OPER_MSI_L0__GR_2BPS_20250801T074015_S20250801T070620_D04_N05.11","bbox":[101.385001349224,79.6835150322236,103.900525677419,80.0894153951246],"type":"Feature","links":[{"rel":"collection","type":"application/json","href":"http://rs-server-catalog:8000/catalog/collections/abutu_SPRINT_34_RSPY_618_TEST_COLLECTION"},{"rel":"parent","type":"application/json","href":"http://rs-server-catalog:8000/catalog/collections/abutu_SPRINT_34_RSPY_618_TEST_COLLECTION"},{"rel":"root","type":"application/json","href":"http://rs-server-catalog:8000/catalog/"},{"rel":"self","type":"application/geo+json","href":"http://rs-server-catalog:8000/catalog/collections/abutu_SPRINT_34_RSPY_618_TEST_COLLECTION/items/S2B_OPER_MSI_L0__GR_2BPS_20250801T074015_S20250801T070620_D04_N05.11"}],"assets":{},"geometry":{"type":"Polygon","coordinates":[[[103.080911931087,80.0894153951246],[101.385001349224,79.8622675367147],[102.217935425109,79.6835150322236],[103.900525677419,79.9060170135641],[103.0809119310

In [15]:
import copy, pystac

COLL = CATALOG_COLLECTION_ID
item_id = "S1A_20210410031928012345"

# pystac.Item
item = catalog_client.get_item(COLL, item_id)

# clone + bad geometry
bad_item = pystac.Item.from_dict(item.to_dict())
bad = bad_item.to_dict()
bad["geometry"] = {"type": "Polygon", "coordinates": "not-an-array"}  # invalid
bad_item = pystac.Item.from_dict(bad)

# PUT via client (400)
try:
    catalog_client.update_item(bad_item)
except Exception as e:
    print("update_item failed:", e)


update_item failed: 400 Client Error: Bad Request for url: http://rs-server-catalog:8000/catalog/collections/abutu:SPRINT_34_RSPY_618_TEST_COLLECTION/items/S1A_20210410031928012345
Detail: Invalid GeoJSON geometry: could not convert string to float: 'n'


In [16]:
result = list(catalog_client.get_collection(CATALOG_COLLECTION_ID).get_items())
result

[<Item id=S2B_OPER_MSI_L0__GR_2BPS_20250801T074015_S20250801T070620_D04_N05.11>,
 <Item id=S1A_20210410031928012345>]

In [17]:
result = catalog_client.remove_collection(CATALOG_COLLECTION_ID)
assert result.json()["deleted collection"] == CATALOG_COLLECTION_ID
pp.pprint(result.json())

{'deleted collection': 'SPRINT_34_RSPY_618_TEST_COLLECTION'}
